<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/master/colab/train_linhe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaEarth-System — Linhe PEFT Training on Colab

Trains Prithvi + PEFT adapters on Linhe RGB patches for two tasks:
- **1.a** Building extraction (2-class, synth label)
- **1.b** Land-use classification (6-class, Esri LULC label)

**Before running:**
1. Upload to Drive root: `linhe_2025_h1.tar.gz`, `linhe_2025_h2.tar.gz` (+ `.sha256`), `linhe_lulc_masks.tar.gz`
2. Runtime -> Change runtime type -> L4 GPU
3. Run cells top-to-bottom (skip cell 8 if 1.a already done)

In [1]:
# 1. Mount Drive, pick archives
from google.colab import drive
drive.mount('/content/drive')

# Two-half packaging keeps each tar.gz under 500 MB so Drive 1 GB free tier
# can hold both simultaneously. Order does not matter — the second tar's
# _index.parquet replaces the first; we merge below in cell 5.
ARCHIVES = [
    'linhe_2025_h1.tar.gz',   # Q1 + Q2  (~420 MB)
    'linhe_2025_h2.tar.gz',   # Q3 + Q4  (~408 MB)
]
CHECKPOINT_DIR = '/content/drive/MyDrive/linhe_ckpt'
RESULTS_DIR = '/content/drive/MyDrive/linhe_results'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted; checkpoint dir:', CHECKPOINT_DIR)

Mounted at /content/drive
Drive mounted; checkpoint dir: /content/drive/MyDrive/linhe_ckpt


In [2]:
# 2. GPU check
!nvidia-smi | head -20

Fri May 15 01:37:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 3. Clone repo (or pull if already cloned)
%cd /content
REPO_URL = 'https://github.com/zhouning/alphaearth-training-system.git'
REPO_DIR = '/content/alphaearth-training-system'
import os, subprocess
if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!git log --oneline -5

/content
/content/alphaearth-training-system
bac033a (HEAD -> master, origin/master, origin/HEAD) paper12: 路 B Section 9 收尾 — abstract/intro 同步 + 配图脚本
7741934 paper12: 路 B Section 9 — Linhe production validation 投 RSE
7c76429 feat: 演示成果 tab — Leaflet 变化热力图 + PEFT 对比 + 41 张像素级对比
78cd15e fix: cell 5 filter LULC index to RGB subset before sanity check
d884df5 refactor: clean notebook for fresh full-run


In [4]:
# 4. Install deps
!pip install -q -e .
!pip install -q pyarrow scikit-learn matplotlib

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for geoadapter (pyproject.toml) ... done


In [5]:
# 5. Extract RGB patches + LULC masks, merge indexes
import shutil, subprocess, os
import pandas as pd

# --- RGB patches (h1 + h2) ---
merged_idx_rows, merged_osm_rows = [], []
for arc in ARCHIVES:
    src = f'/content/drive/MyDrive/{arc}'
    sha_src = src + '.sha256'
    dst = f'/content/{arc}'
    assert os.path.exists(src), f'Upload {arc} to Drive first'
    shutil.copy(src, dst)
    shutil.copy(sha_src, '/content/')
    subprocess.run(['sed', '-i', 's/\r$//', f'/content/{os.path.basename(sha_src)}'], check=True)
    subprocess.run(['sha256sum', '-c', f'/content/{os.path.basename(sha_src)}'],
                   cwd='/content', check=True)
    subprocess.run(['tar', 'xzf', dst, '-C', REPO_DIR + '/'], check=True)
    merged_idx_rows.append(pd.read_parquet('data/linhe_patches/_index.parquet'))
    merged_osm_rows.append(pd.read_parquet('data/linhe_patches/_osm_index.parquet'))
    print(f'  extracted {arc}')

merged_idx = pd.concat(merged_idx_rows, ignore_index=True).drop_duplicates('patch_path')
merged_osm = pd.concat(merged_osm_rows, ignore_index=True).drop_duplicates('patch_path')
# Normalize RGB index paths too
for col in ['patch_path']:
    if col in merged_idx.columns:
        merged_idx[col] = merged_idx[col].str.replace('\\', '/', regex=False)
for col in ['patch_path', 'label_path']:
    if col in merged_osm.columns:
        merged_osm[col] = merged_osm[col].str.replace('\\', '/', regex=False)
merged_idx.to_parquet('data/linhe_patches/_index.parquet')
merged_osm.to_parquet('data/linhe_patches/_osm_index.parquet')
rgb_paths = set(merged_idx['patch_path'])
print(f'RGB patches: {len(merged_idx)} patches, {merged_idx["scene_id"].nunique()} scenes')

# --- LULC masks (1.b) ---
LULC_ARC = 'linhe_lulc_masks.tar.gz'
lulc_src = f'/content/drive/MyDrive/{LULC_ARC}'
if os.path.exists(lulc_src):
    shutil.copy(lulc_src, f'/content/{LULC_ARC}')
    subprocess.run(['tar', 'xzf', f'/content/{LULC_ARC}', '-C', 'data/linhe_patches/'], check=True)
    lulc_idx = pd.read_parquet('data/linhe_patches/_lulc_index.parquet')
    # Normalize paths
    for col in ['patch_path', 'lulc_path']:
        if col in lulc_idx.columns:
            lulc_idx[col] = lulc_idx[col].str.replace('\\', '/', regex=False)
    # Filter to only LULC rows whose RGB patch is in the Colab subset
    n_before = len(lulc_idx)
    lulc_idx = lulc_idx[lulc_idx['patch_path'].isin(rgb_paths)].reset_index(drop=True)
    n_after = len(lulc_idx)
    lulc_idx.to_parquet('data/linhe_patches/_lulc_index.parquet')
    print(f'LULC masks: filtered {n_before} -> {n_after} rows (matching RGB subset)')
    print(f'  years: {sorted(lulc_idx["year"].unique())}')
    # Sanity check: pick a sample that should exist
    sample = lulc_idx.iloc[0]
    assert os.path.exists(sample['patch_path']), f'MISSING RGB: {sample["patch_path"]}'
    assert os.path.exists(sample['lulc_path']), f'MISSING LULC: {sample["lulc_path"]}'
    print(f'  sanity check passed (sample: {sample["scene_id"]} year={sample["year"]})')
else:
    print(f'WARNING: {LULC_ARC} not on Drive, 1.b will not work')

  extracted linhe_2025_h1.tar.gz
  extracted linhe_2025_h2.tar.gz
RGB patches: 21690 patches, 69 scenes
LULC masks: filtered 71840 -> 43380 rows (matching RGB subset)
  years: [np.int64(2022), np.int64(2023)]
  sanity check passed (sample: GF2_PMS1_E107.1_N40.6_20250208_L1A14435155001-PAN1 year=2022)


In [6]:
# 6. Download Prithvi weights from HuggingFace (350 MB)
import os
os.makedirs('data/weights/prithvi', exist_ok=True)
if not os.path.exists('data/weights/prithvi/Prithvi_100M.pt'):
    !wget -q https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M/resolve/main/Prithvi_100M.pt -O data/weights/prithvi/Prithvi_100M.pt
!ls -la data/weights/prithvi/

total 443048
drwxr-xr-x 2 root root      4096 May 15 01:38 .
drwxr-xr-x 3 root root      4096 May 15 01:38 ..
-rw-r--r-- 1 root root 453671052 May 15 01:39 Prithvi_100M.pt


In [7]:
# 7. GPU throughput sanity check (~30 s)
!python scripts/bench_prithvi_throughput.py --steps 20 --batch-size 16

[info] device=cuda, batch=16, patch=128, steps=20
[ok] linear_probe   step=49ms (3.0 ms/patch), trainable=1,538
[ok] houlsby        step=90ms (5.6 ms/patch), trainable=1,191,170

=== Full Linhe demo projection (170 scenes × 700 patches × 20 epoch = 2.38M sample-passes) ===
  linear_probe  : 2.0 h  (119 min)
  houlsby       : 3.7 h  (222 min)

=== 2025Q1-Q4 demo subset (68 scenes × 700 patches × 20 epoch = 952K sample-passes) ===
  linear_probe  : 0.8 h  (48 min)
  houlsby       : 1.5 h  (89 min)


In [8]:
# 8. Run 1.a buildings benchmark (synth baseline)
CONFIG_1A = 'geoadapter/bench/configs/linhe_buildings.yaml'
OUTPUT_1A = f'{RESULTS_DIR}/linhe_buildings.json'
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1A} \
    --output {OUTPUT_1A} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --checkpoint-every 2

Total experiments: 6
Resuming from /content/drive/MyDrive/linhe_results/linhe_buildings.json: 6 experiments already completed
  SKIP linear_probe|rgb_3band|seed=42 (already done)
  SKIP linear_probe|rgb_3band|seed=123 (already done)
  SKIP linear_probe|rgb_3band|seed=456 (already done)
  SKIP houlsby|rgb_3band|seed=42 (already done)
  SKIP houlsby|rgb_3band|seed=123 (already done)
  SKIP houlsby|rgb_3band|seed=456 (already done)
Results saved to /content/drive/MyDrive/linhe_results/linhe_buildings.json


In [ ]:
# 9. Run 1.b LULC 6-class segmentation benchmark
CONFIG_1B = 'geoadapter/bench/configs/linhe_lulc.yaml'
OUTPUT_1B = f'{RESULTS_DIR}/linhe_lulc_seg.json'
CHECKPOINT_DIR_1B = f'{CHECKPOINT_DIR}_lulc'
os.makedirs(CHECKPOINT_DIR_1B, exist_ok=True)
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1B} \
    --output {OUTPUT_1B} \
    --checkpoint-dir {CHECKPOINT_DIR_1B} \
    --checkpoint-every 2

Total experiments: 15
Resuming from /content/drive/MyDrive/linhe_results/linhe_lulc_seg.json: 6 experiments already completed
  SKIP linear_probe|rgb_3band|seed=42 (already done)
  SKIP linear_probe|rgb_3band|seed=123 (already done)
  SKIP linear_probe|rgb_3band|seed=456 (already done)
  [bitfit|rgb_3band|seed=42] device=cuda, trainable_params=107,526
  [bitfit|rgb_3band|seed=42] Loaded linhe_lulc: 3000 train, 600 val
  [bitfit|rgb_3band|seed=42] Epoch 10/30 loss=0.8546
  [bitfit|rgb_3band|seed=42] Epoch 20/30 loss=0.8376
  [bitfit|rgb_3band|seed=42] Epoch 30/30 loss=0.8284
  [bitfit|rgb_3band|seed=42] mIoU=0.1977
  -> appended to /content/drive/MyDrive/linhe_results/linhe_lulc_seg.json (7/15)
  [bitfit|rgb_3band|seed=123] device=cuda, trainable_params=107,526
  [bitfit|rgb_3band|seed=123] Loaded linhe_lulc: 3000 train, 600 val
  [bitfit|rgb_3band|seed=123] Epoch 10/30 loss=0.8581
  [bitfit|rgb_3band|seed=123] Epoch 20/30 loss=0.8346
  [bitfit|rgb_3band|seed=123] Epoch 30/30 loss=0.828

In [ ]:
# 10. Peek results
import json, os
import pandas as pd

all_results = []
for label, path in [('1.a buildings', OUTPUT_1A), ('1.b LULC', OUTPUT_1B)]:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        for r in data:
            r['task'] = label
        all_results.extend(data)

if all_results:
    df = pd.DataFrame(all_results)
    metric_col = 'mIoU' if 'mIoU' in df.columns else None
    show_cols = ['task', 'method', 'seed', 'trainable_params']
    if metric_col:
        show_cols.append(metric_col)
    show_cols = [c for c in show_cols if c in df.columns]
    print(df[show_cols].sort_values(show_cols[:2], ascending=[True, False]).to_string())
else:
    print('No results yet')

## Notes

- **Resume**: Re-running cell 8 or 9 auto-resumes from last checkpoint and skips completed experiments.
- **1.b only**: Skip cell 8 if 1.a is already done (results on Drive).
- **Clean slate**: Delete `linhe_ckpt/`, `linhe_ckpt_lulc/`, and result JSONs on Drive to force full re-run.
- **Budget**: 1.a + 1.b ~ 250 compute units total.